# Lora单卡训练 

In [1]:
import os
import json
import torch
from datasets import Dataset
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer, EarlyStoppingCallback
from peft import LoraConfig, TaskType, get_peft_model
from modelscope import snapshot_download

In [2]:
traindata_path = "../datasets/11/train_test/train0819.jsonl"
evaldata_path = "../datasets/11/train_test/eval0819.jsonl"
model_path = "../models/Qwen2-0.5B-Instruct"
output_path = "../models/Qwen2-0.5B-Instruct_fr_0819"

In [3]:
model_id = "Qwen/Qwen2-0.5B-Instruct"
print("开始从 modelscope下载模型")

snapshot_download(
    model_id=model_id,
    local_dir=model_path,
    # ModelScope 默认就是下载实体文件，不需要特别指定 symlinks 参数
)


开始从 modelscope下载模型


2026-05-23 18:33:28,014 - modelscope - INFO - Target directory already exists, skipping creation.


'../models/Qwen2-0.5B-Instruct'

In [4]:
def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as file:
        data = [json.loads(line) for line in file]
    return pd.DataFrame(data)


In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False, trust_remote_code=True)
tokenizer

Qwen2Tokenizer(name_or_path='../models/Qwen2-0.5B-Instruct', vocab_size=151643, model_max_length=32768, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [6]:
tokenizer("你是谁")

{'input_ids': [105043, 100165], 'attention_mask': [1, 1]}

In [7]:
def preprocess(item, tokenizer, max_length=2048, instruction=None):
    system_message = "You are a helpful assistant."
    instruction = item["instruction"] if instruction is None else instruction
    user_message = instruction + "\n" + item["input"]
    assistant_message = json.dumps({"is_fraud": item["label"]}, ensure_ascii=False)

    message = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": assistant_message},
    ]
    # 
    full_ids = tokenizer.apply_chat_template(
        message,
        tokenize=True,
        add_generation_prompt=False,
        max_length=max_length,
        truncation=True,
    )
    prompt_ids = tokenizer.apply_chat_template(
        message[:-1],
        tokenize=True,
        add_generation_prompt=True,
        max_length=max_length,
        truncation=True,
    )
    full_ids = full_ids["input_ids"]
    prompt_ids = prompt_ids["input_ids"]
    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
    }


In [8]:
def load_dataset(train_path, eval_path, tokenizer):
    train_df = load_jsonl(train_path)
    train_ds = Dataset.from_pandas(train_df)
    train_dataset = train_ds.map(
        lambda x: preprocess(x, tokenizer),
        remove_columns=train_ds.column_names,
        desc="Tokenizing train dataset",
    )

    eval_df = load_jsonl(eval_path)
    eval_ds = Dataset.from_pandas(eval_df)
    eval_dataset = eval_ds.map(
        lambda x: preprocess(x, tokenizer),
        remove_columns=eval_ds.column_names,
        desc="Tokenizing eval dataset",
    )

    return train_dataset, eval_dataset

In [9]:
train_dataset, eval_dataset = load_dataset(traindata_path, evaldata_path, tokenizer)

Tokenizing train dataset:   0%|          | 0/18787 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/2348 [00:00<?, ? examples/s]

In [10]:
print(train_dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 18787
})


In [11]:
print(
    f"Input IDs: {train_dataset[0]['input_ids']}\n"
    f"Attention Mask: {train_dataset[0]['attention_mask']}\n"
    f"Labels: {train_dataset[0]['labels']}"
)

Input IDs: [151644, 8948, 198, 2610, 525, 264, 10950, 17847, 13, 151645, 198, 151644, 872, 271, 100431, 99639, 37474, 105051, 108704, 11, 220, 14880, 101042, 105051, 43815, 107189, 106037, 101052, 3837, 23031, 2236, 68805, 66017, 103929, 104317, 59151, 9623, 761, 97957, 25, 830, 91233, 8, 3407, 110395, 18, 25, 10236, 236, 108, 102865, 101393, 99487, 101314, 100006, 101189, 100006, 85336, 99360, 102683, 99225, 106630, 104528, 3837, 85336, 26939, 99487, 104671, 100634, 20412, 104917, 100634, 99557, 104366, 115203, 99487, 108398, 100634, 99650, 104468, 3837, 99650, 99725, 100662, 99792, 99692, 46944, 46944, 104160, 32757, 8997, 110395, 16, 25, 58230, 109, 20412, 151645, 198, 151644, 77091, 198, 4913, 285, 761, 97957, 788, 895, 92, 151645, 198]
Attention Mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

In [12]:

tokenizer.decode(train_dataset[0]['input_ids'], skip_special_tokens=True)

'system\nYou are a helpful assistant.\nuser\n\n下面是一段对话文本, 请分析对话内容是否有诈骗风险，以json格式输出你的判断结果(is_fraud: true/false)。\n\n发言人3: 现在我所在这个哪里能够工艺能够去把屈光做得很好的，去到这个省级医院是自治区医院跟广西医科大学这个附属医院他们还可以，他们一直保持比较好的一个一个手术量。\n发言人1: 就是\nassistant\n{"is_fraud": false}\n'

In [13]:
# 
tokenizer.decode(list(filter(lambda x: x != -100, train_dataset[0]["labels"])))

'{"is_fraud": false}<|im_end|>\n'

In [14]:
device = "cuda:0"

In [15]:
def load_model(model_path, device='cuda'):
    model = AutoModelForCausalLM.from_pretrained(model_path,torch_dtype=torch.bfloat16)
    model.enable_input_require_grads() # 开启梯度检查点时，要执行该方法
    return model.to(device)

model = load_model(model_path, device)
model

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/root/miniconda3/lib/python3.12/site-packages/torch/cuda/__init__.py:235: UserWarning: 
NVIDIA GeForce RTX 5090 with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA GeForce RTX 5090 GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

## 插入微调矩阵

使用Lora进行微调时，需要修改模型结构，这里将一个rank=8的低秩矩阵插入到模型的每个DecodeLayer层中，在训练时只学习这个低秩矩阵，原模型的参数不改变。

1. target_modules：定义了要对模型的哪些块做修改，准确来说是在具体哪些块中插入低秩矩阵。
2. r: 低秩矩阵的秩大小，值越小，模型能学习的参数越少，这里使用默认的8.
3. lora_alpha： 一个缩放比例因子，控制着模型推理过程中将LoRA参数在模型整个参数中所占的比重大小，这里也按推荐配置为r的2倍。
4. lora_dropout: 训练过程中，随机丢弃的神经元比例，目的是引入随机性来增强模型的泛化能力。

In [19]:
def build_peft_model(model):
    config = LoraConfig(
        task_type=TaskType.CAUSAL_LM, 
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        inference_mode=False, # 训练模式
        r=8, 
        lora_alpha=16,   
        lora_dropout=0.05
    )
    return get_peft_model(model, config)

peft_model = build_peft_model(model)
peft_model

RuntimeError: CUDA error: no kernel image is available for execution on the device
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
